In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

In [ ]:
# load 3 datasets and concatenate them
df1 = pd.read_csv('../data/general/news.csv')
df1['label'] = df1['label'].map({'REAL': 0, 'FAKE': 1}) # map labels to 0 and 1
df2 = pd.read_csv('../data/general/True.csv')
df2['label'] = 0 # all entries in this dataset are real news, so label is 0
df3 = pd.read_csv('../data/general/Fake.csv')
df3['label'] = 1 # all entries in this dataset are fake news, so label is 1
df = pd.concat([df1, df2, df3], ignore_index=True)
df.sample(n=5)

,index,title,text,label,subject,date
33907,NaN,Obama Tells Reporters That He Gives Zero F*ck...,Things got a little heated during a press conf...,1,News,"May 27, 2016"
24427,NaN,Polish ruling party and president say closer t...,WARSAW (Reuters) - Poland s ruling PiS party a...,0,worldnews,"October 7, 2017"
12552,NaN,Trump to name ex-Georgia Governor Perdue as ag...,WASHINGTON (Reuters) - U.S. President-elect Do...,0,politicsNews,"January 19, 2017"
41931,NaN,"CLASSLESS, BLACK PANTHER DIVA, Beyonce Shows U...","New Black Panther Entertainer of The Year, Bey...",1,politics,"Apr 2, 2016"
30621,NaN,‘No Real Logic’: French Translators Having A ...,Reality show star turned president Donald Trum...,1,News,"January 24, 2017"


In [ ]:
# drop columns that are not needed for classification
df = df.drop(columns=['index', 'title', 'subject', 'date'])

# remove entries where the text length is <200, as they are likely to be noise
df['text_length'] = df['text'].apply(len)
df = df[df['text_length'] >= 200].drop(columns=['text_length'])

In [ ]:
# since we are partially using ISOT dataset, we need to perform some minor preprocessing to remove 
# the source of the news from the text, as it can be a strong signal for classification and can lead to overfitting. 
# We will remove any URLs and email addresses from the text.

def clean_text(text):
    # 1. Remove Publisher tags (ISOT specific)
    text = re.sub(r'^.*? - ', '', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # 3. Standard cleaning
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s!\?]', '', text) # Keep text, spaces, !, and ?
    return text

df['text'] = df['text'].apply(clean_text)

In [ ]:
# split data into train and test sets
X, y = df['text'], df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# create a pipeline with TfidfVectorizer and SGDClassifier, with hyperparameter weights already tuned previously using GridSearchCV
pipeline = make_pipeline(
    TfidfVectorizer(max_features=10000, ngram_range=(1,2), stop_words='english'),
    SGDClassifier(loss='hinge', penalty=None, alpha=1e-4, random_state=42, learning_rate='pa1', eta0=1.0)
)

In [ ]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))